## 1.生成对应变量的sample_ch4_parameter.nml

In [ ]:
import itertools
import os
import subprocess
import glob, os, shutil, sys
import numpy as np
import pandas as pd
import xarray as xr
import time

def make_namelist(nml_input,nml_output,sample,j):
    # open nml file and readlines
    with open(nml_input+f'standard_ch4_parameter.nml', 'r') as file:
        nml_content = file.readlines()

    # modify var
    for i, line in enumerate(nml_content):
        # replace CASE NAME
        if 'DEF_CH4%q10ch4' in line:
            nml_content[i] = f"DEF_CH4%q10ch4 = {sample['q10ch4']}\n"
        if 'DEF_CH4%f_ch4' in line:
            nml_content[i] = f"DEF_CH4%f_ch4 = {sample['f_ch4']}\n"
        if 'DEF_CH4%vmax_ch4_oxid' in line:
            nml_content[i] = f"DEF_CH4%vmax_ch4_oxid = {sample['vmax_ch4_oxid']}\n"
        if 'DEF_CH4%vmax_oxid_unsat' in line:
            nml_content[i] = f"DEF_CH4%vmax_oxid_unsat = {sample['vmax_oxid_unsat']}\n"
        if 'DEF_CH4%k_m' in line:
            nml_content[i] = f"DEF_CH4%k_m = {sample['k_m']}\n"
        if 'DEF_CH4%k_m_unsat' in line:
            nml_content[i] = f"DEF_CH4%k_m_unsat = {sample['k_m_unsat']}\n"
        if 'DEF_CH4%k_m_o2' in line:
            nml_content[i] = f"DEF_CH4%k_m_o2 = {sample['k_m_o2']}\n"
        if 'DEF_CH4%q10_ch4_oxid' in line:
            nml_content[i] = f"DEF_CH4%q10_ch4_oxid = {sample['q10_ch4_oxid']}\n"
        if 'DEF_CH4%vgc_max' in line:
            nml_content[i] = f"DEF_CH4%vgc_max = {sample['vgc_max']}\n"
        if 'DEF_CH4%poros_tiller' in line:
            nml_content[i] = f"DEF_CH4%poros_tiller = {sample['poros_tiller']}\n"
        if 'DEF_CH4%poros_tiller_unsat' in line:
            nml_content[i] = f"DEF_CH4%poros_tiller_unsat = {sample['poros_tiller_unsat']}\n"
        if 'DEF_CH4%aere_radius' in line:
            nml_content[i] = f"DEF_CH4%aere_radius = {sample['aere_radius']}\n"
        if 'DEF_CH4%rob' in line:
            nml_content[i] = f"DEF_CH4%rob = {sample['rob']}\n"
        if 'DEF_CH4%scale_factor_aere' in line:
            nml_content[i] = f"DEF_CH4%scale_factor_aere = {sample['scale_factor_aere']}\n"
        if 'DEF_CH4%scale_factor_gasdiff' in line:
            nml_content[i] = f"DEF_CH4%scale_factor_gasdiff = {sample['scale_factor_gasdiff']}\n"
        if 'DEF_CH4%scale_factor_liqdiff' in line:
            nml_content[i] = f"DEF_CH4%scale_factor_liqdiff = {sample['scale_factor_liqdiff']}\n"

    # read back modified nml file
    # sometimes need modify nml file path

    new_file_path = f"{nml_output}sample{j+1}_ch4_parameter.nml"
    print(new_file_path)
    with open(new_file_path, 'w') as file:
        file.writelines(nml_content)

if __name__ == '__main__':
    # mode='no_spin_up'
    # forcing ='FLUXNET-CH4'
    samplelist = f"/share/home/dq076/mode/ME/251030_r/run/scaled_samples.csv"
    samplelists = pd.read_csv(samplelist, header=0)
    nml_input = "/share/home/dq076/mode/ME/251030_r/run/"
    nml_output = f"{nml_input}site/samples/"
    os.makedirs(nml_output, exist_ok=True)
    samplelists['vmax_oxid_unsat'] = samplelists['vmax_ch4_oxid']/10
    samplelists['k_m_unsat'] = samplelists['k_m']/10
    samplelists['poros_tiller_unsat'] = samplelists['poros_tiller']/6
    samplelists['scale_factor_liqdiff'] = samplelists['scale_factor_gasdiff']

    for j in range(len(samplelists)):
        sample = samplelists.iloc[j]
        # print(sample)
        make_namelist(nml_input,nml_output,sample,j)

/share/home/dq076/mode/ME/251030_r/run/site/samples/sample1_ch4_parameter.nml
/share/home/dq076/mode/ME/251030_r/run/site/samples/sample2_ch4_parameter.nml
/share/home/dq076/mode/ME/251030_r/run/site/samples/sample3_ch4_parameter.nml
/share/home/dq076/mode/ME/251030_r/run/site/samples/sample4_ch4_parameter.nml
/share/home/dq076/mode/ME/251030_r/run/site/samples/sample5_ch4_parameter.nml
/share/home/dq076/mode/ME/251030_r/run/site/samples/sample6_ch4_parameter.nml
/share/home/dq076/mode/ME/251030_r/run/site/samples/sample7_ch4_parameter.nml
/share/home/dq076/mode/ME/251030_r/run/site/samples/sample8_ch4_parameter.nml
/share/home/dq076/mode/ME/251030_r/run/site/samples/sample9_ch4_parameter.nml
/share/home/dq076/mode/ME/251030_r/run/site/samples/sample10_ch4_parameter.nml
/share/home/dq076/mode/ME/251030_r/run/site/samples/sample11_ch4_parameter.nml
/share/home/dq076/mode/ME/251030_r/run/site/samples/sample12_ch4_parameter.nml
/share/home/dq076/mode/ME/251030_r/run/site/samples/sample13_

## 2.制作对应forcing，mode，sample的运行nml

In [ ]:
import os
import pandas as pd
from pathlib import Path

def get_timestamp_info_fast(station_list, path='/share/home/dq076/data/ME/FLUXNET-CH4/'):
    """快速提取CSV时间戳信息 - 使用行读取而非pandas"""
    site = f'FLX_{station_list["SITE_ID"]}_FLUXNET-CH4_{station_list["YEAR_START"]}-{station_list["YEAR_END"]}_1-1'
    hh_name = f'FLX_{station_list["SITE_ID"]}_FLUXNET-CH4_HH_{station_list["YEAR_START"]}-{station_list["YEAR_END"]}_1-1.csv'
    csv_path = f"{path}{site}/{hh_name}"
    
    # 直接用文件读取，避免pandas开销
    with open(csv_path, 'r') as f:
        header = f.readline().strip().split(',')
        first_line = f.readline().strip().split(',')
        
        # 找到TIMESTAMP列的索引
        try:
            ts_start_idx = header.index('TIMESTAMP_START')
            ts_end_idx = header.index('TIMESTAMP_END')
        except ValueError:
            raise ValueError(f"找不到TIMESTAMP列: {csv_path}")
        
        # 读取最后一行
        f.seek(0, 2)  # 移到文件末尾
        file_size = f.tell()
        f.seek(max(0, file_size - 1024))  # 往回读1KB（足够一行）
        lines = f.readlines()
        last_line = lines[-1].strip().split(',')
    
    # 确定使用哪个时间戳
    first_ts_start = first_line[ts_start_idx]
    if first_ts_start[:4] == str(station_list['YEAR_START']):
        start = first_ts_start
        end = last_line[ts_start_idx]
    else:
        start = first_line[ts_end_idx]
        end = last_line[ts_end_idx]
    
    return {
        'start_second': int(start[8:10]) * 3600 + int(start[10:12]) * 60,
        'end_second': int(end[8:10]) * 3600 + int(end[10:12]) * 60,
        'start_month': int(start[4:6]),
        'start_day': int(start[6:8]),
        'end_month': int(end[4:6]),
        'end_day': int(end[6:8])
    }

def create_namelist_fast(template_lines, station_info, sample, forcing, mode, nml_input):
    """使用字符串模板和单次遍历优化"""
    
    # 预构建替换字典（键为要查找的字符串片段）
    replacements = {
        "DEF_CASE_NAME = ": f"DEF_CASE_NAME = '{station_info['SITE_ID']}'\n",
        "DEF_simulation_time%start_year = ": f"DEF_simulation_time%start_year = {station_info['YEAR_START']}\n",
        "DEF_simulation_time%start_month = ": f"DEF_simulation_time%start_month = {station_info['start_month']}\n",
        "DEF_simulation_time%start_day = ": f"DEF_simulation_time%start_day = {station_info['start_day']}\n",
        "DEF_simulation_time%start_sec = ": f"DEF_simulation_time%start_sec = {station_info['start_second']}\n",
        "DEF_simulation_time%end_year = ": f"DEF_simulation_time%end_year = {station_info['YEAR_END']}\n",
        "DEF_simulation_time%end_month = ": f"DEF_simulation_time%end_month = {station_info['end_month']}\n",
        "DEF_simulation_time%end_day = ": f"DEF_simulation_time%end_day = {station_info['end_day']}\n",
        "DEF_simulation_time%end_sec = ": f"DEF_simulation_time%end_sec = {station_info['end_second']}\n",
        "DEF_simulation_time%spinup_year = ": f"DEF_simulation_time%spinup_year = {int(station_info['YEAR_START']) - 1}\n",
        "SITE_fsitedata = ": f"SITE_fsitedata = '{station_info['srfpath']}'\n",
        "DEF_dir_output = ": f"DEF_dir_output = '/share/home/dq076/data/cases/v2/site/{forcing}/{mode}/{sample}/'\n",
        "DEF_file_METHANE_para = ": f"DEF_file_METHANE_para = '{nml_input}site/samples/{sample}_ch4_parameter.nml'\n",
        "DEF_forcing_namelist = ": f"DEF_forcing_namelist = '{nml_input}/site/{forcing}/forcing/SINGLE_{station_info['SITE_ID']}.nml'\n"
    }
    
    # 单次遍历，使用startswith检查（比in更快）
    result = []
    for line in template_lines:
        replaced = False
        for key, value in replacements.items():
            if line.lstrip().startswith(key.split('=')[0].strip()):
                result.append(value)
                replaced = True
                break
        if not replaced:
            result.append(line)
    
    return result

def batch_create_directories(base_path, num_samples):
    """批量创建目录，避免重复检查"""
    dirs_to_create = [f"{base_path}sample{i+1}/" for i in range(num_samples)]
    for d in dirs_to_create:
        Path(d).mkdir(parents=True, exist_ok=True)
    return dirs_to_create

def main():
    mode = 'no_spin_up'
    forcing = 'FLUXNET-CH4'
    
    stnlist = "/share/home/dq076/data/ME/FLUXNET-CH4/FLX_AA-Flx_CH4-META_wetland.csv"
    station_lists = pd.read_csv(stnlist, header=0)
    station_lists = station_lists[station_lists['FLUXNET-CH4_DATA_POLICY'] == 'CCBY4.0'].reset_index(drop=True)
    station_lists = station_lists[station_lists['SITE_ID'] != 'HK-MPM'].reset_index(drop=True)
    nml_input = "/share/home/dq076/mode/ME/251030_r/run/"
    
    # 读取模板文件一次
    template_path = f"{nml_input}US-Los_{forcing}_{mode}.nml"
    with open(template_path, 'r') as f:
        template_lines = f.readlines()
    
    # 添加srfpath
    station_lists['srfpath'] = station_lists.apply(
        lambda row: f'/share/home/dq076/data/CoLM_Forcing/PLUMBER2/Srfdata/{row["SITE_ID"]}_{row["YEAR_START"]}-{row["YEAR_END"]}_FLUXNET-CH4_Srf.nc',
        axis=1
    )
    
    n = len(station_lists)
    print(f"处理 {n} 个站点，每个站点120个样本，共 {n * 120} 个任务")
    
    # 输出基础路径
    nml_output_base = f"{nml_input}site/{forcing}/{mode}/"
    
    # 批量创建所有目录（一次性完成）
    print("批量创建目录...")
    all_dirs = batch_create_directories(nml_output_base, 120)
    
    # 处理每个站点
    total_count = 0
    for idx in range(n):
        station_dict = station_lists.iloc[idx].to_dict()
        
        # 获取时间戳信息（每个站点一次）
        try:
            timestamp_info = get_timestamp_info_fast(station_dict)
        except Exception as e:
            print(f"错误: 站点 {station_dict['SITE_ID']} 读取CSV失败: {e}")
            continue
        
        # 合并站点信息和时间戳信息
        station_info = {**station_dict, **timestamp_info}
        
        # 为该站点生成120个样本
        for sample_idx in range(120):
            sample = f'sample{sample_idx + 1}'
            output_path = f"{all_dirs[sample_idx]}{station_dict['SITE_ID']}.nml"
            
            # 生成内容
            content = create_namelist_fast(template_lines, station_info, sample, forcing, mode, nml_input)
            
            # 写入文件
            with open(output_path, 'w') as f:
                f.writelines(content)
            
            total_count += 1
        
        # 显示进度
        if (idx + 1) % 5 == 0 or (idx + 1) == n:
            print(f"进度: {idx + 1}/{n} 个站点完成 ({total_count} 个文件)")
    
    print(f"\n完成! 总共生成 {total_count} 个namelist文件")

if __name__ == '__main__':
    import time
    start_time = time.time()
    main()
    elapsed = time.time() - start_time
    print(f"总耗时: {elapsed:.2f} 秒")

处理 44 个站点，每个站点120个样本，共 5280 个任务
批量创建目录...
进度: 5/44 个站点完成 (600 个文件)
进度: 10/44 个站点完成 (1200 个文件)
进度: 15/44 个站点完成 (1800 个文件)
进度: 20/44 个站点完成 (2400 个文件)
进度: 25/44 个站点完成 (3000 个文件)
进度: 30/44 个站点完成 (3600 个文件)
进度: 35/44 个站点完成 (4200 个文件)
进度: 40/44 个站点完成 (4800 个文件)
进度: 44/44 个站点完成 (5280 个文件)

完成! 总共生成 5280 个namelist文件
总耗时: 12.42 秒


## 3.批量运行120*44次CoLM计算

In [ ]:
import subprocess
import os
from joblib import Parallel, delayed
import glob
from concurrent.futures import ProcessPoolExecutor, as_completed  # 切换到 ProcessPoolExecutor 以支持实时进度

def load_environment(env_file):
    cmd = f'bash -c "source {env_file} && env"'
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    new_env = {}
    for line in result.stdout.strip().split('\n'):
        if '=' in line:
            key, value = line.split('=', 1)
            if not key.startswith('BASH_FUNC_'):
                new_env[key] = value
    
    os.environ.update(new_env)
    return os.environ.copy()

def run_colm(run_path, nml_path, log_path, nml_name, updated_env):
    """
    处理单个 nml 文件，返回成功/失败状态（不打印进度，由主脚本处理）。
    """
    nml_file = f'{nml_path}{nml_name}.nml'
    log_file = f'{log_path}{nml_name}.txt'
    
    try:
        # 用 'w' 模式打开文件，重置内容
        with open(log_file, 'w', encoding='utf-8') as log:
            log.write(f"=== 处理 {nml_name}.nml ===\n")
            log.flush()
            
            commands = [
                [f'{run_path}mksrfdata.x', nml_file],
                [f'{run_path}mkinidata.x', nml_file],
                [f'{run_path}colm.x', nml_file]
            ]
            
            for cmd in commands:
                log.write(f"执行命令: {' '.join(cmd)}\n")
                log.flush()
                
                subprocess.run(cmd, 
                               env=updated_env, 
                               stdout=log, 
                               stderr=subprocess.STDOUT, 
                               text=True)
                
                log.write("\n" + "="*50 + "\n")
                log.flush()
            
            log.write(f"=== {nml_name} 处理完成 ===\n")
            log.flush()
        
        return True  # 成功
    except Exception as e:
        # 如果失败，也记录到日志
        with open(log_file, 'w', encoding='utf-8') as log:
            log.write(f"=== {nml_name}.nml 处理失败: {str(e)} ===\n")
        return False

if __name__ == "__main__":
    for j in range(120):
        sample = f'sample{j+1}'
        forcing ='FLUXNET-CH4'
        mode ='no_spin_up'

        env_file = '/share/home/dq089/soft/gnu-env'
        run_path = '/share/home/dq076/mode/ME/251030_r/run/'

        nml_path = f'{run_path}site/{forcing}/{mode}/{sample}/'
        log_path = f'{nml_path}logs/'  
        os.makedirs(log_path, exist_ok=True)

        updated_env = load_environment(env_file)
        
        nml_files = glob.glob(f'{nml_path}*.nml')
        nml_names = [os.path.splitext(os.path.basename(nml_file))[0] for nml_file in nml_files if 'HK-MPM' not in nml_file]
        print(f"发现 {len(nml_files)} 个 .nml 文件：{nml_names}")

        max_workers = min(24, os.cpu_count() or 1)
        with ProcessPoolExecutor(max_workers=max_workers) as executor:
            # 提交所有任务，返回 future 对象
            future_to_nml = {
                executor.submit(run_colm, run_path, nml_path, log_path, nml_name, updated_env): nml_name
                for nml_name in nml_names
            }
            
            # 维护剩余任务集合
            remaining_nml = set(nml_names)
            completed_count = 0
            
            # 实时监控完成
            for future in as_completed(future_to_nml):
                nml_name = future_to_nml[future]
                try:
                    success = future.result()
                    if success:
                        completed_count += 1
                    remaining_nml.discard(nml_name)  # 移除已完成（无论成功/失败）
                    
                    # 打印进度：总数 + 剩余列表
                    print(f"=== {nml_name} 处理完成（成功: {success}) ===")
                    print(f"已完成总数: {completed_count}/{len(nml_names)}")
                    if remaining_nml:
                        print(f"剩余未处理: {sorted(list(remaining_nml))}")
                    else:
                        print("所有任务已完成！")
                    print("-" * 50)
                    
                except Exception as exc:
                    print(f"{nml_name} 执行异常: {exc}")
                    remaining_nml.discard(nml_name)
                    completed_count += 1  # 视作完成（失败）
                    print(f"已完成总数: {completed_count}/{len(nml_names)}")                                                                                                                                                                                                                                 
                    if remaining_nml:
                        print(f"剩余未处理: {sorted(list(remaining_nml))}")
                    print("-" * 50)
        
        print("批量处理结束。")